# NIRF Rankings 2017-2025: Merge, Check and Standardize

This notebook merges the 9 yearly NIRF files, finds the data problems, and fixes them.
After **every code cell** there is a short note saying what the cell did and why.

**Columns in the data**

| Column | Meaning |
|---|---|
| `year`, `category` | Ranking year and category (Overall, Engineering, ...) |
| `rank`, `score` | Rank and total score |
| `institute_name`, `city`, `state` | Institute details |
| `institute_id` | ID given by NIRF (changes between years) |
| `tlr`, `rpc`, `go`, `oi`, `pr` | Parameter scores: Teaching-Learning, Research, Graduation Outcomes, Outreach, Perception |

In [3]:
import pandas as pd
import re

**Cell 1 - Imports**
- `pandas` for the tables, `re` for cleaning text.

In [4]:
files = [
    "dataset/nirf_2017.csv",
    "dataset/nirf_2018.csv",
    "dataset/nirf_2019.csv",
    "dataset/nirf_2020.csv",
    "dataset/nirf_2021.csv",
    "dataset/nirf_2022.csv",
    "dataset/nirf_2023.csv",
    "dataset/nirf_2024.csv",
    "dataset/nirf_2025.csv",
]

dfs = []
for f in files:
    dfs.append(pd.read_csv(f, encoding="utf-8-sig"))   # utf-8-sig removes the hidden BOM

df = pd.concat(dfs, ignore_index=True)
print(df.shape)
df.head()

(6924, 13)


,year,category,rank,institute_name,city,state,score,institute_id,tlr,rpc,go,oi,pr
0,2017,Overall,1.0,Indian Institute of Science Bangalore,Bengaluru,Karnataka,83.28,IR17-I-2-18243,83.11,87.59,87.97,61.48,83.33
1,2017,Overall,2.0,Indian Institute of Technology Madras,Chennai,Tamil Nadu,73.97,IR17-I-1-1-77,69.49,72.60,84.02,76.75,68.70
2,2017,Overall,3.0,Indian Institute of Technology Bombay,Mumbai,Maharashtra,71.78,IR17-I-2-18633,64.68,78.14,70.07,65.80,83.44
3,2017,Overall,4.0,Indian Institute of Technology Kharagpur,Kharagpur,West Bengal,68.43,IR17-I-2-18630,55.07,70.46,85.11,72.85,64.72
4,2017,Overall,5.0,Indian Institute of Technology Delhi,New Delhi,Delhi,64.18,IR17-I-2-1-79,55.45,68.48,65.92,68.69,69.53


**Cell 2 - Load and merge**
- Reads each CSV and stacks them into one table with `pd.concat`.
- `encoding="utf-8-sig"` is needed because every file starts with a hidden BOM character that would otherwise rename the first column.
- All 9 files have the same 13 columns, so they stack cleanly.

In [5]:
df.info()
print()
print(df.isnull().sum())

<class 'pandas.DataFrame'>
RangeIndex: 6924 entries, 0 to 6923
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   year            6924 non-null   int64  
 1   category        6924 non-null   str    
 2   rank            6920 non-null   float64
 3   institute_name  6924 non-null   str    
 4   city            6924 non-null   str    
 5   state           6924 non-null   str    
 6   score           6923 non-null   float64
 7   institute_id    6924 non-null   str    
 8   tlr             6673 non-null   float64
 9   rpc             6673 non-null   float64
 10  go              6673 non-null   float64
 11  oi              6923 non-null   float64
 12  pr              6923 non-null   float64
dtypes: float64(7), int64(1), str(5)
memory usage: 1.2 MB

year                0
category            0
rank                4
institute_name      0
city                0
state               0
score               1
institute_id    

**Cell 3 - First look at types and missing values**
- `rank` is a float (`1.0`) because a few ranks are missing.
- Missing values exist in `rank`, `score`, `tlr`, `rpc`, `go`, `oi`, `pr` (explained in Cells 7-8).

In [6]:
# remove extra spaces
for c in ["institute_name", "city", "state"]:
    df[c] = df[c].str.strip()

# rank: whole number, but keep missing values as <NA>
df["rank"] = df["rank"].astype("Int64")

df.dtypes

year                int64
category              str
rank                Int64
institute_name        str
city                  str
state                 str
score             float64
institute_id          str
tlr               float64
rpc               float64
go                float64
oi                float64
pr                float64
dtype: object

**Cell 4 - Basic cleaning**
- Trims spaces in the text columns.
- Converts `rank` to a whole number. `Int64` (capital I) is used because it can still hold missing values, which normal `int` cannot.

In [7]:
print("Full row duplicates:                 ", df.duplicated().sum())
print("Same year + category + id:            ", df.duplicated(["year", "category", "institute_id"]).sum())
print("Same year + category + name:          ", df.duplicated(["year", "category", "institute_name"]).sum())
print("Same year + category + name + city:   ", df.duplicated(["year", "category", "institute_name", "city"]).sum())

ranked = df.dropna(subset=["rank"])
print("Same year + category + rank:          ", ranked.duplicated(["year", "category", "rank"]).sum())

Full row duplicates:                  0
Same year + category + id:             0
Same year + category + name:           81
Same year + category + name + city:    0
Same year + category + rank:           168


**Cell 5 - Count every type of duplicate**

| Check | Result | Meaning |
|---|---|---|
| Full row / same id | 0 | No true duplicates |
| Same name | 81 | Looks like duplicates, but see Cell 6 |
| Same name + city | 0 | So the 81 are different institutes in different cities |
| Same rank | 168 | Ties, see Cell 7 |

In [8]:
same_name = df[df.duplicated(["year", "category", "institute_name"], keep=False)]
same_name = same_name.sort_values(["institute_name", "year", "category"])
same_name[["year", "category", "rank", "institute_name", "city", "state"]]

,year,category,rank,institute_name,city,state
2288,2020,Pharmacy,58,Bharati Vidyapeeth's College of Pharmacy,Navi Mumbai,Maharashtra
2292,2020,Pharmacy,62,Bharati Vidyapeeth's College of Pharmacy,Kolhapur,Maharashtra
1568,2019,Pharmacy,49,Bharati Vidyapeeth`s College of Pharmacy,Navi Mumbai,Maharashtra
1573,2019,Pharmacy,54,Bharati Vidyapeeth`s College of Pharmacy,Kolhapur,Maharashtra
1697,2019,Medical,3,Christian Medical College,Vellore,Tamil Nadu
...,...,...,...,...,...,...
2052,2020,Engineering,97,University College of Engineering,Kakinada,Andhra Pradesh
5933,2024,Agriculture,11,University of Agricultural Sciences,Bengaluru,Karnataka
5946,2024,Agriculture,24,University of Agricultural Sciences,Dharwad,Karnataka
921,2018,College,22,Women`s Christian College,Chennai,Tamil Nadu


**Cell 6 - Look at the same-name cases**
- Each pair has the same name but a **different city**, e.g. *Holy Cross College* in Tiruchirappalli and in Kanyakumari.
- They are separate institutes, so they must **not** be dropped.
- Cell 11 keeps them apart by adding the city to the key.

In [9]:
ties = ranked[ranked.duplicated(["year", "category", "rank"], keep=False)]
ties = ties.sort_values(["year", "category", "rank"])
print("Rows involved in ties:", len(ties))

missing_rank = df[df["rank"].isna()]
missing_rank[["year", "category", "rank", "institute_name", "score"]]

Rows involved in ties: 331


,year,category,rank,institute_name,score
721,2018,Engineering,<NA>,Indian Institute of Engineering Science and Te...,53.24
727,2018,Engineering,<NA>,Indian Institute of Technology Mandi,51.28
866,2018,Pharmacy,<NA>,Banasthali Vidyapith,53.41
902,2018,College,<NA>,Bishop Heber College,67.63


**Cell 7 - Rank ties and missing ranks**
- **Ties:** two institutes got the same score and share one rank. These are real, so they stay.
- NIRF does not skip the next rank after a tie, so a category can have more rows than its highest rank (e.g. 2020 Overall has 101 rows but ranks only go to 100).
- **Missing ranks:** 4 rows, all from 2018 (2 Engineering, 1 Pharmacy, 1 College). The source file has no rank for them. They are left as `<NA>` because guessing a rank could be wrong.

In [10]:
# 1) one row with no score and no parameters at all
print(df[df["score"].isna()][["year", "category", "rank", "institute_name"]])

# 2) which categories have missing tlr / rpc / go ?
print()
print(df[df["tlr"].isna()].groupby(["year", "category"]).size())

      year category  rank                                     institute_name
6761  2025   Dental    18  Faculty of Dental Sciences, Institute of Medic...

year  category
2021  Research    50
2022  Research    50
2023  Research    50
2024  Research    50
2025  Dental       1
      Research    50
dtype: int64


**Cell 8 - Missing scores**
- **2025 Dental, rank 18** (Faculty of Dental Sciences, IMS Varanasi): `score` and all 5 parameter columns are empty in the source.
- **Research category (2021-2025, 250 rows):** `tlr`, `rpc` and `go` are empty for every row. Only `oi` and `pr` are filled, so these parameters are simply not given for this category.
- These are **left as NaN**. Filling them with 0 or an average would give wrong scores.

In [11]:
def clean_text(text):
    text = text.lower().replace("&", " and ")
    text = re.sub(r"[^a-z0-9 ]", " ", text)       # removes . , ' ` ( ) - etc.
    return re.sub(r"\s+", " ", text).strip()      # removes extra spaces

def clean_city(city):
    city = re.sub(r"\b(district|dist)\b", "", clean_text(city))   # "Dist. Calicut" -> "calicut"
    return re.sub(r"\s+", " ", city).strip()

# same place written in different ways
city_fix = {
    "bangalore": "bengaluru", "mysore": "mysuru", "gurgaon": "gurugram",
    "calicut": "kozhikode", "trichur": "thrissur", "belgaum": "belagavi",
    "mangalore": "mangaluru", "mengaluru": "mangaluru",
    "tiruchirapalli": "tiruchirappalli", "allahabad": "prayagraj",
}
state_fix = {"pondicherry": "puducherry"}

df["name_clean"] = df["institute_name"].apply(clean_text)
df["city_clean"] = df["city"].apply(clean_city).replace(city_fix)
df["state_clean"] = df["state"].apply(clean_text).replace(state_fix)

df[["institute_name", "name_clean", "city_clean", "state_clean"]].sample(5, random_state=1)

,institute_name,name_clean,city_clean,state_clean
6545,Sri Guru Gobind Singh College of Commerce,sri guru gobind singh college of commerce,delhi,delhi
5066,Banaras Hindu University,banaras hindu university,varanasi,uttar pradesh
4302,Thapar Institute of Engineering and Technology...,thapar institute of engineering and technology...,patiala,punjab
3144,"S.B. College, Changanacherry",s b college changanacherry,kottayam,kerala
5866,Government Dental College,government dental college,ahmedabad,gujarat


**Cell 9 - Cleaning functions**
- Lower-case, remove punctuation (so `` St. Joseph`s `` and `St. Joseph's` match), remove "Dist./District", and fix "&" vs "and".
- `city_fix` joins spelling variants (Bangalore / Bengaluru, Calicut / Kozhikode, ...).
- `state_fix` joins `Pondicherry` and `Puducherry`. Lower-casing also joins `Jammu And Kashmir` and `Jammu and Kashmir`.
- The clean values go in **new columns**, so the original columns are untouched for now.

In [12]:
# Some names have the city added after a comma: "Indian Institute of Science, Bengaluru"
# Remove that city part, but only if the name without it is already used somewhere else in the same state.
known = set(zip(df["name_clean"], df["state_clean"]))

def drop_city_suffix(name, city):
    m = re.search(r",\s*" + re.escape(city) + r"\s*$", name, flags=re.I)
    return name[:m.start()] if m else name

base = pd.Series([drop_city_suffix(n, c) for n, c in zip(df["institute_name"], df["city"])]).apply(clean_text)
use_base = [(b, s) in known for b, s in zip(base, df["state_clean"])]

changed = (base != df["name_clean"]) & pd.Series(use_base)
print("Names fixed:", changed.sum())

df["name_clean"] = base.where(use_base, df["name_clean"])

Names fixed: 289


**Cell 10 - Names with the city added**
- Some years write `Indian Institute of Science, Bengaluru` and other years write `Indian Institute of Science`. They are the same institute.
- The city part is removed **only when** the shorter name already exists in the same state. This avoids damaging names like *University of Delhi*.

In [13]:
# key = name + state  (state is safer than city because city spellings change a lot)
df["key"] = df["name_clean"] + " | " + df["state_clean"]

# if one key appears twice in the same year + category, they are different institutes -> add city
clash = df[df.duplicated(["year", "category", "key"], keep=False)]["key"].unique()
print("Names that clash:", len(clash))
for c in sorted(clash):
    print(" -", c)

mask = df["key"].isin(clash)
df.loc[mask, "key"] = df["key"] + " | " + df["city_clean"]

# one new id per institute, same in every year
df["inst_id"] = df["key"].factorize()[0] + 1

Names that clash: 12
 - bharati vidyapeeth s college of pharmacy | maharashtra
 - government arts college | tamil nadu
 - government dental college | kerala
 - holy cross college | tamil nadu
 - indian institute of management | maharashtra
 - jaipuria institute of management | uttar pradesh
 - kasturba medical college | karnataka
 - kle college of pharmacy | karnataka
 - manipal college of dental sciences | karnataka
 - st joseph s college | kerala
 - university of agricultural sciences | karnataka
 - women s christian college | tamil nadu


**Cell 11 - Build one stable `inst_id` per institute**
- The old `institute_id` changes between years (2,144 different IDs for 1,318 names), so it cannot track an institute over time.
- New key = cleaned name + state. This joins the spelling variants in Cells 9-10.
- The clash list shows the names that are really different institutes in the same state (e.g. *Government Dental College* in Kozhikode and Thiruvananthapuram). For only these, city is added to the key.
- `factorize()` turns each key into a number: `inst_id`.

In [14]:
# use the latest year's spelling for name, city and state, so each institute has ONE version
df = df.sort_values(["year", "category", "rank"]).reset_index(drop=True)

for c in ["institute_name", "city", "state"]:
    df[c] = df.groupby("inst_id")[c].transform("last")

df[df["institute_name"].str.contains("Indian Institute of Science")][["inst_id", "institute_name", "city", "state"]].drop_duplicates()

,inst_id,institute_name,city,state
250,1,Indian Institute of Science Bangalore,Bengaluru,Karnataka
278,29,Indian Institute of Science Education & Resear...,Pune,Maharashtra
293,44,Indian Institute of Science Education & Resear...,Mohanpur,West Bengal
301,52,Indian Institute of Science Education & Resear...,Mohali,Punjab
347,98,Indian Institute of Science Education & Resear...,Bhopal,Madhya Pradesh
795,357,Indian Institute of Science,Bengaluru,Karnataka
852,367,Indian Institute of Science Education & Resear...,Thiruvananthapuram,Kerala


**Cell 12 - One standard name, city and state per institute**
- Every row of an institute now uses its **latest year's** name, city and state.
- This removes the leftover variants: `Bangalore` vs `Bengaluru`, backticks in names, `Pondicherry` vs `Puducherry`, and names with a city added.

In [15]:
print("Unique institutes:", df["inst_id"].nunique())
print("Duplicates left (year + category + inst_id):", df.duplicated(["year", "category", "inst_id"]).sum())
print("Names per inst_id (max):", df.groupby("inst_id")["institute_name"].nunique().max())
print("States:", df["state"].nunique())
print()
print("How many years each institute appears in:")
print(df.groupby("inst_id")["year"].nunique().value_counts().sort_index())

Unique institutes: 1124
Duplicates left (year + category + inst_id): 0
Names per inst_id (max): 1
States: 33

How many years each institute appears in:
year
1    311
2    171
3     96
4    110
5     90
6     78
7     48
8     89
9    131
Name: count, dtype: int64


**Cell 13 - Verify**
- **0 duplicates** on year + category + `inst_id`, so each institute appears once per category per year.
- Each `inst_id` has exactly one name.
- The last table shows how many institutes appear in 1 year, 2 years, ... 9 years.

In [16]:
final = df[["inst_id", "institute_name", "city", "state", "year", "category",
            "rank", "score", "tlr", "rpc", "go", "oi", "pr"]]
final = final.sort_values(["year", "category", "rank"]).reset_index(drop=True)

final.to_csv("nirf_merged.csv", index=False)
print("Saved:", final.shape)
final.head()

Saved: (6924, 13)


,inst_id,institute_name,city,state,year,category,rank,score,tlr,rpc,go,oi,pr
0,257,Miranda House,Delhi,Delhi,2017,College,1,69.39,75.76,27.44,90.06,96.45,41.82
1,258,Loyola College,Chennai,Tamil Nadu,2017,College,2,68.68,66.92,100.00,48.42,56.45,80.45
2,259,Shri Ram College of Commerce,Delhi,Delhi,2017,College,3,67.18,77.63,0.20,97.51,93.80,53.98
3,260,Bishop Heber College,Tiruchirappalli,Tamil Nadu,2017,College,4,61.18,76.62,53.99,61.28,68.11,18.63
4,261,Atma Ram Sanatan Dharma College,New Delhi,Delhi,2017,College,5,60.68,74.22,30.54,79.60,82.51,0.21


**Cell 14 - Final table and save**
- Keeps 13 clean columns: institute info, `year`, `category`, `rank`, `score` and the 5 parameter scores.
- The helper columns (`name_clean`, `key`, ...) and the old `institute_id` are dropped.
- Saved as `nirf_merged.csv`.

# Summary: All Issues, Reasons and Solutions

## A. Duplicate-looking cases

| # | Issue | Why it happens | Solution |
|---|---|---|---|
| 1 | **Same name appears twice** in one year and category (81 rows, 26 names) | **Different institutes** with the same name in different cities, e.g. Holy Cross College in Tiruchirappalli and in Kanyakumari | **Do not drop.** Add city to the key only for the 12 names that clash (Cell 11) |
| 2 | **Same rank appears twice** (168 rows) | **Genuine ties.** Two institutes have the same score, so they share a rank | **Keep them** |
| 3 | **Full-row and same-ID duplicates** | None exist (0 found) | Nothing to do |

## B. Inconsistent values

| # | Issue | Why it happens | Solution |
|---|---|---|---|
| 4 | **`institute_id` changes between years** (2,144 IDs for 1,318 names) | NIRF changed its ID format over the years (`IR17-...`, `IR-1-O-O-U-...`, `IR-O-U-...`) | Make our own `inst_id` from name + state (Cell 11) |
| 5 | **Backtick vs apostrophe** `` St. Joseph`s `` vs `St. Joseph's` (201 names) | Different typing in different years | `clean_text()` removes punctuation (Cell 9) |
| 6 | **Commas and city added to names** `Indian Institute of Science, Bengaluru` vs `Indian Institute of Science` (650 names have commas) | Naming style changed between years | Remove punctuation, and remove the city part when the shorter name already exists (Cells 9-10) |
| 7 | **City spelling varies** Bangalore / Bengaluru, Calicut / Kozhikode, Mysore / Mysuru | Cities were renamed or written in different ways | `city_fix` dictionary; state used in the key instead of city (Cell 9) |
| 8 | **"Dist." / "District" in cities** | Source formatting | `clean_city()` removes them (Cell 9) |
| 9 | **State spelling** `Pondicherry` (2017-2020) vs `Puducherry` (2021+); `Jammu And Kashmir` (2017) vs `Jammu and Kashmir` | Official rename and capitalization | `state_fix` and lower-casing (Cell 9) |
| 10 | **Same name, same state, different institutes** (12 names, e.g. Government Dental College in Kerala) | Real separate institutes | City added to the key for these only (Cell 11) |

## C. Missing values

| # | Issue | Why it happens | Solution |
|---|---|---|---|
| 11 | **4 missing ranks** (all 2018: 2 Engineering, 1 Pharmacy, 1 College) | Rank is empty in the source file | Left as `<NA>` using `Int64`; not guessed (Cell 4) |
| 12 | **1 row with no score and no parameters** (2025 Dental, rank 18) | Values are empty in the source file | Left as NaN (Cell 8) |
| 13 | **`tlr`, `rpc`, `go` empty for all 250 Research rows** (2021-2025) | Not given for this category; only `oi` and `pr` are filled | Left as NaN. Do not fill with 0 (Cell 8) |

## D. Formatting issues

| # | Issue | Why it happens | Solution |
|---|---|---|---|
| 14 | **Hidden BOM character** breaks the first column name | Files saved as UTF-8 with BOM | `encoding="utf-8-sig"` (Cell 2) |
| 15 | **Rank is a float** (`1.0`) in 2017-2020 | Different export format and the missing values | `astype("Int64")` (Cell 4) |

## E. Things to know (not errors)

| # | Note | Explanation |
|---|---|---|
| 16 | **Some categories have 101 rows** (2020 Overall, 2022 Management, 2022 Pharmacy, 2025 Pharmacy) or fewer ranks than rows (2019 Overall, 2024 Law) | Ties do not skip the next rank, so one extra row appears |
| 17 | **Engineering has 200 rows in 2019 and 2020**, 100 in other years | NIRF listed more institutes in those two years |
| 18 | **Categories were added over time:** Law, Architecture, Medical (2018), Dental (2020), Research (2021), Agriculture (2023), State Public University (2024) | New NIRF categories. Do not compare a category across years before it existed |
| 19 | **The same institute appears in several categories in one year** (Overall + Engineering) | Normal, not a duplicate |

## F. Limits that remain

- Spelling mistakes in names (e.g. "Enterprenurship") and short forms like "(IIITDM)" are not fixed, so a few institutes may still be split in two.
- For the 12 same-name-same-state institutes, city is part of the key, so a new city spelling not in `city_fix` could split one of them.

## Order of the steps

1. **Load** (Cells 1-2) -> 2. **Check** (Cell 3) -> 3. **Basic clean** (Cell 4) -> 4. **Inspect duplicates and missing values** (Cells 5-8) -> 5. **Standardize and build `inst_id`** (Cells 9-12) -> 6. **Verify and save** (Cells 13-14)